In [25]:
import nibabel as nib
import numpy as np
from scipy.ndimage import median_filter
from skimage import exposure
from skimage.restoration import denoise_bilateral
from scipy.ndimage import zoom


input_file = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE-rotated-gaze-left/ACF1.rot.44f4.fr.rothschild.S.22887586.1_00000.DCM.nii"
output_file = input_file[:-4] + "_cropped_denoised.nii"

In [26]:
def crop_nifti(input_path, output_path, zoom_factor, x_range, y_range, z_range, kernel_size):
    """
    Crop a 3D NIfTI image and save it with updated affine.

    Parameters:
        input_path (str): input .nii/.nii.gz
        output_path (str): output file
        x_range, y_range, z_range: tuples (start, end)
    """

    nii = nib.load(input_path)
    data = np.asarray(nii.dataobj)  # efficient loading
    affine = nii.affine.copy()
    header = nii.header.copy()

    print(data.shape)

    # Crop data
    cropped = data[
        x_range[0]:x_range[1],
        y_range[0]:y_range[1],
        z_range[0]:z_range[1]
    ]


    # --- Normalize (important before CLAHE) ---
    c_min, c_max = cropped.min(), cropped.max()
    norm = (cropped - c_min) / (c_max - c_min + 1e-8)

    # --- CLAHE (slice-by-slice) ---
    equalized = np.zeros_like(norm)

    for z in range(norm.shape[2]):
        equalized[:, :, z] = exposure.equalize_adapthist(
            norm[:, :, z],
            clip_limit=0.015  # tweak this
        )

        equalized[:, :, z] = denoise_bilateral(
            equalized[:, :, z],
            sigma_color=0.1,
            sigma_spatial=3
        )

    # --- Rescale back to original intensity range ---
    equalized = equalized * (c_max - c_min) + c_min

    # Optional: restore dtype
    equalized = equalized.astype(nii.get_data_dtype())

    equalized = median_filter(equalized, size=kernel_size)


    # --- Update affine ---
    # Compute offset in voxel space
    offset = np.array([x_range[0], y_range[0], z_range[0], 1])

    # Convert voxel offset to world coordinates
    new_origin = affine @ offset

    # Update translation part of affine
    new_affine = affine.copy()
    new_affine[:3, 3] = new_origin[:3]

    if zoom_factor > 1:
        # --- Resample ---
        zoomed = zoom(equalized, zoom=zoom_factor, order=3)  # cubic interpolation

        # --- Update affine ---
        new_affine[:3, :3] /= zoom_factor  # voxel size becomes smaller
        equalized = zoomed




    # Create new NIfTI
    cropped_nii = nib.Nifti1Image(equalized, new_affine, header)

    # Save
    nib.save(cropped_nii, output_path)

    print(f"Cropped and denoised volume saved to: {output_path}")


crop_nifti(input_file, output_file, 1, [80, 170], [40,160], [10,90], 2)


(320, 320, 112)
Cropped and denoised volume saved to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE-rotated-gaze-left/ACF1.rot.44f4.fr.rothschild.S.22887586.1_00000.DCM_cropped_denoised.nii


In [27]:
# def denoise_nifti(input_path, output_path, kernel_size=3):
#     """
#     Load a 3D NIfTI file, apply filtering, and save it back.

#     Parameters:
#         input_path : Path to input .nii or .nii.gz file
#         output_path : Path to save denoised file
#         kernel_size : Size of median filter (e.g., 3 or (3,3,3))
#     """

#     # Load NIfTI file
#     nii = nib.load(input_path)

#     # Get data as numpy array
#     data = nii.get_fdata()

#     # Apply median filter (3D)
#     denoised_data = median_filter(data, size=kernel_size)

#     # Preserve affine and header
#     affine = nii.affine
#     header = nii.header

#     # Create new NIfTI image
#     denoised_nii = nib.Nifti1Image(denoised_data, affine, header)

#     # Save output
#     nib.save(denoised_nii, output_path)

#     print(f"Denoised file saved to: {output_path}")



# denoise_nifti(input_file, output_file, kernel_size=3)